# Doc to CSV

In [1]:
import os
import pandas as pd
from docx import Document
import json

def extract_json_from_docx(docx_path):
    document = Document(docx_path)
    json_text = ""
    
    for paragraph in document.paragraphs:
        json_text += paragraph.text.strip()
    
    try:
        return json.loads(json_text)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON in {docx_path}: {e}")
        return None

def normalize_flight_data(json_data):
    data = []
    for record in json_data:
        flat_record = {
            "type": record.get("type"),
            "status": record.get("status"),
            # Departure details
            "departure_iata": record.get("departure", {}).get("iataCode"),
            "departure_icao": record.get("departure", {}).get("icaoCode"),
            "departure_terminal": record.get("departure", {}).get("terminal"),
            "departure_scheduled": record.get("departure", {}).get("scheduledTime"),
            "departure_estimated": record.get("departure", {}).get("estimatedTime"),
            "departure_actual": record.get("departure", {}).get("actualTime"),
            "departure_runway_estimated": record.get("departure", {}).get("estimatedRunway"),
            "departure_runway_actual": record.get("departure", {}).get("actualRunway"),
            # Arrival details
            "arrival_iata": record.get("arrival", {}).get("iataCode"),
            "arrival_icao": record.get("arrival", {}).get("icaoCode"),
            "arrival_terminal": record.get("arrival", {}).get("terminal"),
            "arrival_scheduled": record.get("arrival", {}).get("scheduledTime"),
            "arrival_estimated": record.get("arrival", {}).get("estimatedTime"),
            # Airline details
            "airline_name": record.get("airline", {}).get("name"),
            "airline_iata": record.get("airline", {}).get("iataCode"),
            "airline_icao": record.get("airline", {}).get("icaoCode"),
            # Flight details
            "flight_number": record.get("flight", {}).get("number"),
            "flight_iata": record.get("flight", {}).get("iataNumber"),
            "flight_icao": record.get("flight", {}).get("icaoNumber"),
        }
        data.append(flat_record)
    return pd.DataFrame(data)

def process_docx_directory(directory_path):
    #Process all DOCX files in a directory and convert them into a single DataFrame.
    all_data = pd.DataFrame()  

    for file_name in os.listdir(directory_path):
        if file_name.endswith(".docx"):
            file_path = os.path.join(directory_path, file_name)
            print(f"Processing {file_name}...")
            
            json_data = extract_json_from_docx(file_path)
            if json_data:
                df = normalize_flight_data(json_data)
                all_data = pd.concat([all_data, df], ignore_index=True)

    return all_data

directory_path = "Test"  

combined_df = process_docx_directory(directory_path)

if not combined_df.empty:
    print(combined_df.head())  
    
    csv_path = "test_combined.csv"  
    combined_df.to_csv(csv_path, index=False)
    print(f"CSV saved at {csv_path}")
else:
    print("No data found in DOCX files.")


Processing 1.docx...
Processing 10.docx...
Processing 11.docx...
Processing 12.docx...
Processing 13.docx...
Processing 14.docx...
Processing 15.docx...
Processing 16.docx...
Processing 17.docx...
Processing 18.docx...
Processing 19.docx...
Processing 2.docx...
Processing 20.docx...
Processing 21.docx...
Processing 22.docx...
Processing 23.docx...
Processing 24.docx...
Processing 25.docx...
Processing 26.docx...
Processing 27.docx...
Processing 28.docx...
Processing 29.docx...
Processing 3.docx...
Processing 30.docx...
Processing 31.docx...
Processing 32.docx...
Processing 33.docx...
Processing 34.docx...
Processing 35.docx...
Processing 36.docx...
Processing 37.docx...
Processing 39.docx...
Processing 4.docx...
Processing 40.docx...
Processing 41.docx...
Processing 42.docx...
Processing 43.docx...
Processing 44.docx...
Processing 45.docx...
Processing 46.docx...
Processing 47.docx...
Processing 48.docx...
Processing 49.docx...
Processing 5.docx...
Processing 50.docx...
Processing 51.d

# Preprocessing

In [12]:
f=pd.read_csv('test_combined.csv')

In [13]:
f.head()

,type,status,departure_iata,departure_icao,departure_terminal,departure_scheduled,departure_estimated,departure_actual,departure_runway_estimated,departure_runway_actual,...,arrival_icao,arrival_terminal,arrival_scheduled,arrival_estimated,airline_name,airline_iata,airline_icao,flight_number,flight_iata,flight_icao
0,departure,active,lhe,opla,NaN,2023-07-17t20:35:00.000,NaN,NaN,2023-07-17t20:46:00.000,2023-07-17t20:46:00.000,...,opkc,NaN,2023-07-17t22:20:00.000,2023-07-17t22:12:00.000,flyjinnah,9p,fjl,847,9p847,fjl847
1,departure,active,lhe,opla,m,2023-07-27t08:00:00.000,NaN,NaN,NaN,NaN,...,oerk,t2,2023-07-27t10:00:00.000,NaN,pakistan international airlines,pk,pia,725,pk725,pia725
2,departure,active,lhe,opla,m,2023-07-27t08:00:00.000,NaN,NaN,NaN,NaN,...,omdb,1,2023-07-27t10:00:00.000,NaN,ethiopian airlines,et,eth,4359,et4359,eth4359
3,departure,unknown,lhe,opla,m,2023-07-28t16:45:00.000,NaN,NaN,NaN,NaN,...,oejn,NaN,2023-07-28t20:30:00.000,NaN,airblue,pa,abq,470,pa470,abq470
4,departure,active,lhe,opla,m,2023-07-19t04:15:00.000,NaN,NaN,2023-07-19t04:18:00.000,2023-07-19t04:18:00.000,...,omaa,3,2023-07-19t06:35:00.000,2023-07-19t06:08:00.000,klm,kl,klm,3932,kl3932,klm3932


In [14]:
df=f.drop(['type','departure_terminal','arrival_terminal', 'flight_number', 'flight_iata', 'flight_icao'], axis=1)

In [15]:
def datetime(df):
  return pd.to_datetime(df)

In [16]:
df['departure_actual']=df['departure_actual'].apply(datetime)
df['departure_runway_estimated']=df['departure_runway_estimated'].apply(datetime)
df['departure_runway_actual']=df['departure_runway_actual'].apply(datetime)
df['arrival_scheduled']=df['arrival_scheduled'].apply(datetime)
df['arrival_estimated']=df['arrival_estimated'].apply(datetime)

df['departure_scheduled']=df['departure_scheduled'].apply(datetime)
df['departure_estimated']=df['departure_estimated'].apply(datetime)

In [17]:
df['year'] = df['departure_scheduled'].dt.year
df['month'] = df['departure_scheduled'].dt.month
df['day'] = df['departure_scheduled'].dt.day

In [18]:
df.head()

,status,departure_iata,departure_icao,departure_scheduled,departure_estimated,departure_actual,departure_runway_estimated,departure_runway_actual,arrival_iata,arrival_icao,arrival_scheduled,arrival_estimated,airline_name,airline_iata,airline_icao,year,month,day
0,active,lhe,opla,2023-07-17 20:35:00,NaT,NaT,2023-07-17 20:46:00,2023-07-17 20:46:00,khi,opkc,2023-07-17 22:20:00,2023-07-17 22:12:00,flyjinnah,9p,fjl,2023,7,17
1,active,lhe,opla,2023-07-27 08:00:00,NaT,NaT,NaT,NaT,ruh,oerk,2023-07-27 10:00:00,NaT,pakistan international airlines,pk,pia,2023,7,27
2,active,lhe,opla,2023-07-27 08:00:00,NaT,NaT,NaT,NaT,dxb,omdb,2023-07-27 10:00:00,NaT,ethiopian airlines,et,eth,2023,7,27
3,unknown,lhe,opla,2023-07-28 16:45:00,NaT,NaT,NaT,NaT,jed,oejn,2023-07-28 20:30:00,NaT,airblue,pa,abq,2023,7,28
4,active,lhe,opla,2023-07-19 04:15:00,NaT,NaT,2023-07-19 04:18:00,2023-07-19 04:18:00,auh,omaa,2023-07-19 06:35:00,2023-07-19 06:08:00,klm,kl,klm,2023,7,19


In [19]:
df.shape

(14910, 18)

In [20]:
df['departure_delay']=df['departure_scheduled']-df['departure_actual']

In [21]:
df.head()

,status,departure_iata,departure_icao,departure_scheduled,departure_estimated,departure_actual,departure_runway_estimated,departure_runway_actual,arrival_iata,arrival_icao,arrival_scheduled,arrival_estimated,airline_name,airline_iata,airline_icao,year,month,day,departure_delay
0,active,lhe,opla,2023-07-17 20:35:00,NaT,NaT,2023-07-17 20:46:00,2023-07-17 20:46:00,khi,opkc,2023-07-17 22:20:00,2023-07-17 22:12:00,flyjinnah,9p,fjl,2023,7,17,NaT
1,active,lhe,opla,2023-07-27 08:00:00,NaT,NaT,NaT,NaT,ruh,oerk,2023-07-27 10:00:00,NaT,pakistan international airlines,pk,pia,2023,7,27,NaT
2,active,lhe,opla,2023-07-27 08:00:00,NaT,NaT,NaT,NaT,dxb,omdb,2023-07-27 10:00:00,NaT,ethiopian airlines,et,eth,2023,7,27,NaT
3,unknown,lhe,opla,2023-07-28 16:45:00,NaT,NaT,NaT,NaT,jed,oejn,2023-07-28 20:30:00,NaT,airblue,pa,abq,2023,7,28,NaT
4,active,lhe,opla,2023-07-19 04:15:00,NaT,NaT,2023-07-19 04:18:00,2023-07-19 04:18:00,auh,omaa,2023-07-19 06:35:00,2023-07-19 06:08:00,klm,kl,klm,2023,7,19,NaT


In [22]:
df.to_csv('test.csv',index=False)